# 08 · Integration Testing & Snapshots

**Focus:** exercising a real FastAPI app end-to-end, and regression-testing large responses with `syrupy`.

Two ideas combine here:

1. **Integration testing with `TestClient`.** FastAPI ships a `TestClient` (built on `httpx`) that
   calls your app **in-process** — no server to start, no ports, no network. You get a normal HTTP
   response object back and assert on it.
2. **Snapshot testing with `syrupy`.** When a response is a large, deeply nested JSON blob, writing
   `assert response == {...huge dict...}` by hand is painful and error-prone. syrupy's `snapshot`
   fixture records the value on first run (into a `__snapshots__/` file) and, on every later run,
   asserts the value still matches. It's a regression tripwire for structured data.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### A FastAPI app returning deep nested JSON
An endpoint that assembles a realistic, nested user-profile payload.

In [2]:
%%writefile nb08_app.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/profile/{user_id}")
def profile(user_id: int):
    return {
        "user": {
            "id": user_id,
            "name": "Ada Lovelace",
            "roles": ["admin", "author"],
            "settings": {
                "theme": "dark",
                "notifications": {"email": True, "sms": False, "push": True},
            },
        },
        "stats": {"posts": 42, "followers": 1337, "following": {"users": 12, "topics": 5}},
        "recent": [
            {"id": 1, "title": "Notes on the Analytical Engine"},
            {"id": 2, "title": "On Testing"},
        ],
    }

Writing nb08_app.py


### The integration test
`TestClient(app)` calls the app in-process. We assert on the status code and a couple of key fields,
then hand the **entire** JSON body to `snapshot` — syrupy handles the deep comparison.

In [3]:
%%writefile test_nb08.py
from fastapi.testclient import TestClient
from nb08_app import app

client = TestClient(app)

def test_profile_status_and_fields():
    resp = client.get("/profile/7")
    assert resp.status_code == 200
    body = resp.json()
    assert body["user"]["id"] == 7
    assert body["user"]["settings"]["notifications"]["push"] is True

def test_profile_matches_snapshot(snapshot):
    resp = client.get("/profile/7")
    assert resp.json() == snapshot        # syrupy records/compares the whole payload

Writing test_nb08.py


### Step 1 — create the snapshot
The **first** time, there's no stored snapshot, so we run with `--snapshot-update` to record it.
syrupy writes it into a `__snapshots__/` folder next to the test. You'll see "1 snapshot generated".

In [4]:
!pytest -v --snapshot-update test_nb08.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 2 items                                                              

test_nb08.py::test_profile_status_and_fields PASSED                      [ 50%]
test_nb08.py::test_profile_matches_snapshot PASSED                       [100%]

=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/fastapi/testclient.py:1
  /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
--------------------------- snapshot report summary ----------------------------
1 snapshot generated.
========================= 2 passed, 1 warning in 0.35s =========================


### Step 2 — assert against the stored snapshot
Now run **normally**. syrupy compares the live response to what it recorded. Because the app is
unchanged, the snapshot matches and the test passes. If someone later changed a field, this test
would fail and show a diff — that's the regression tripwire.

In [5]:
!pytest -v test_nb08.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 2 items                                                              

test_nb08.py::test_profile_status_and_fields PASSED                      [ 50%]
test_nb08.py::test_profile_matches_snapshot PASSED                       [100%]

=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/fastapi/testclient.py:1
  /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
--------------------------- snapshot report summary ----------------------------
1 snapshot passed.
========================= 2 passed, 1 warning in 0.29s =========================


### Peek at what syrupy stored
The snapshot is human-readable — you can review it in code review like any other file.

In [6]:
!cat __snapshots__/test_nb08.ambr

# serializer version: 1
# name: test_profile_matches_snapshot
  dict({
    'recent': list([
      dict({
        'id': 1,
        'title': 'Notes on the Analytical Engine',
      }),
      dict({
        'id': 2,
        'title': 'On Testing',
      }),
    ]),
    'stats': dict({
      'followers': 1337,
      'following': dict({
        'topics': 5,
        'users': 12,
      }),
      'posts': 42,
    }),
    'user': dict({
      'id': 7,
      'name': 'Ada Lovelace',
      'roles': list([
        'admin',
        'author',
      ]),
      'settings': dict({
        'notifications': dict({
          'email': True,
          'push': True,
          'sms': False,
        }),
        'theme': 'dark',
      }),
    }),
  })
# ---


### ⚠️ Common pitfall — don't reflexively `--snapshot-update`
A snapshot only protects you if you *review* changes. Running `--snapshot-update` on every failure
just rubber-stamps whatever the code now returns, silently blessing regressions. Update snapshots only
after you've confirmed the new output is correct. (Also: snapshots break on non-deterministic fields
like timestamps or random ids — exclude those.)

### 🔬 Try it yourself — watch the tripwire fire
We now change **one** value in the API (`followers`) and re-run the *existing* snapshot test **without**
`--snapshot-update`. **Predict:** what happens? Run it and read the diff syrupy prints — that diff is
exactly what a code reviewer would see when a response accidentally changes.

In [7]:
%%writefile nb08_app.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/profile/{user_id}")
def profile(user_id: int):
    return {
        "user": {
            "id": user_id,
            "name": "Ada Lovelace",
            "roles": ["admin", "author"],
            "settings": {
                "theme": "dark",
                "notifications": {"email": True, "sms": False, "push": True},
            },
        },
        "stats": {"posts": 42, "followers": 9999, "following": {"users": 12, "topics": 5}},
        "recent": [
            {"id": 1, "title": "Notes on the Analytical Engine"},
            {"id": 2, "title": "On Testing"},
        ],
    }

Overwriting nb08_app.py


In [8]:
!pytest -v test_nb08.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... 

collected 2 items                                                              

test_nb08.py::test_profile_status_and_fields PASSED                      [ 50%]
test_nb08.py::test_profile_matches_snapshot 

FAILED                       [100%]

=================================== FAILURES ===================================
________________________ test_profile_matches_snapshot _________________________

snapshot = dict({
  'recent': list([
    dict({
      'id': 1,
      'title': 'Notes on the Analytical Engine',
    }),
    dict(...({
        'email': True,
        'push': True,
        'sms': False,
      }),
      'theme': 'dark',
    }),
  }),
})

    def test_profile_matches_snapshot(snapshot):
        resp = client.get("/profile/7")
>       assert resp.json() == snapshot        # syrupy records/compares the whole payload
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       AssertionError: assert [+ received] == [- snapshot]
E           dict({
E            ...
E             'stats': dict({
E         -     'followers':...
E         
E         ...Full output truncated (5 lines hidden), use '-vv' to show

test_nb08.py:15: AssertionError
=============================== warnings summary ======

See the `followers: 1337 -> 9999` diff? The snapshot caught a change we didn't intend. In real life
you'd now decide: *bug* (fix the code) or *intended* (review, then `--snapshot-update`).

### What you learned
- FastAPI's `TestClient` exercises your app **in-process** — real routing, no server/network.
- Assert on `resp.status_code` and `resp.json()` for targeted checks.
- syrupy's `snapshot` fixture records a value once (`--snapshot-update`) and guards it forever after.
- Snapshots shine for large/nested payloads where hand-written asserts are impractical.

Next up: **end-to-end browser testing** with Playwright.